<a href="https://colab.research.google.com/github/Biel07-max/Projeto-analise-de-dados/blob/main/analise_dos_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install openpyxl plotly

import pandas as pd
import plotly.express as px
from google.colab import files
from IPython.display import display, Markdown


In [ ]:
print("Selecione o arquivo Excel (.xlsx) da pesquisa:")
arquivo = files.upload()

nome_arquivo = next(
    (nome for nome in arquivo if nome.endswith(".xlsx")),
    None
)

if nome_arquivo is None:
    raise ValueError("Envie um arquivo Excel com extensão .xlsx.")

df = pd.read_excel(nome_arquivo)

Selecione o arquivo Excel (.xlsx) da pesquisa:


Saving Tecnologias e Diretrizes para tratamento e preven��o da Covid-19_P�gina 1_Tabela (1).xlsx to Tecnologias e Diretrizes para tratamento e preven��o da Covid-19_P�gina 1_Tabela (1).xlsx


In [ ]:
df.columns = df.columns.str.strip()

col_decisao = "Decisão do Ministério da Saúde"
col_inicial = "Análise inicial da Conitec"
col_final = "Análise Final da Conitec"
col_demandante = "Demandante"

colunas_necessarias = [
    col_decisao, col_inicial, col_final,
    col_demandante, "Consulta Pública",
    "Contribuições de experiência e opinião",
    "Contribuições técnico - científicas",
    "Relatórios para a Sociedade",
    "Notas Técnicas", "Despacho"
]

for coluna in colunas_necessarias:
    if coluna not in df.columns:
        raise ValueError(f"Coluna não encontrada: {coluna}")

def limpar_coluna(coluna):
    return (
        df[coluna]
        .fillna("Não informado")
        .astype(str)
        .str.strip()
        .replace({"-": "Não informado", "": "Não informado"})
    )

for coluna in colunas_necessarias:
    df[coluna] = limpar_coluna(coluna)

total = len(df)

display(Markdown("#  Análise de dados COVID-19"))
display(Markdown(
    f"**Total de registros analisados:** {total}"
))


#  Análise de dados COVID-19

**Total de registros analisados:** 20

In [ ]:
display(Markdown("## Indicadores gerais"))



indicadores = pd.DataFrame({
    "Indicador": [
        "Total de tecnologias/diretrizes",
        "Registros com decisão do Ministério da Saúde",
        "Registros com consulta pública",
        "Registros com contribuições de experiência e opinião",
        "Registros com contribuições técnico-científicas"
    ],
    "Quantidade": [
        total,
        (df[col_decisao] != "Não informado").sum(),
        (df["Consulta Pública"] != "Não informado").sum(),
        (df["Contribuições de experiência e opinião"] != "Não informado").sum(),
        (df["Contribuições técnico - científicas"] != "Não informado").sum()
    ]
})

display(indicadores)


## Indicadores gerais

,Indicador,Quantidade
0,Total de tecnologias/diretrizes,20
1,Registros com decisão do Ministério da Saúde,19
2,Registros com consulta pública,19
3,Registros com contribuições de experiência e o...,13
4,Registros com contribuições técnico-científicas,13


In [ ]:
display(Markdown("## Decisões do Ministério da Saúde"))

decisoes = (
    df[col_decisao]
    .value_counts()
    .reset_index()
)
decisoes.columns = ["Decisão", "Quantidade"]

fig1 = px.bar(
    decisoes,
    x="Decisão",
    y="Quantidade",
    color="Decisão",
    text="Quantidade",
    title="Distribuição das decisões do Ministério da Saúde"
)


fig1.update_layout(
    xaxis_title="Tipo de decisão",
    yaxis_title="Quantidade de registros",
    showlegend=False,
    xaxis_tickangle=-25
)
fig1.show()

## Decisões do Ministério da Saúde

In [ ]:
display(Markdown("## Proporção das decisões"))

fig2 = px.pie(
    decisoes,
    names="Decisão",
    values="Quantidade",
    title="Proporção de cada tipo de decisão",
    hole=0.35
)
fig2.show()

## Proporção das decisões

In [ ]:
display(Markdown("## Comparação das análises da Conitec"))

comparacao = pd.concat([
    df[col_inicial].value_counts().rename("Análise inicial"),
    df[col_final].value_counts().rename("Análise final")
], axis=1).fillna(0).reset_index()

comparacao = comparacao.rename(columns={"index": "Recomendação"})

comparacao_longa = comparacao.melt(
    id_vars="Recomendação",
    var_name="Etapa",
    value_name="Quantidade"
)




fig3 = px.bar(
    comparacao_longa,
    x="Recomendação",
    y="Quantidade",
    color="Etapa",
    barmode="group",
    text="Quantidade",
    title="Análise inicial x análise final da Conitec"
)



fig3.update_layout(
    xaxis_title="Tipo de recomendação",
    yaxis_title="Quantidade",
    xaxis_tickangle=-25
)
fig3.show()



## Comparação das análises da Conitec

In [ ]:
display(Markdown("## Consultas públicas"))



consultas = pd.DataFrame({
    "Situação": ["Com consulta pública", "Sem informação"],
    "Quantidade": [
        (df["Consulta Pública"] != "Não informado").sum(),
        (df["Consulta Pública"] == "Não informado").sum()
    ]
})


fig4 = px.bar(
    consultas,
    x="Situação",
    y="Quantidade",
    color="Situação",
    text="Quantidade",
    title="Registros de consulta pública"
)
fig4.update_layout(showlegend=False)
fig4.show()


## Consultas públicas

In [ ]:
display(Markdown("##Participação e contribuições"))

participacao = pd.DataFrame({
    "Tipo de contribuição": [
        "Experiência e opinião",
        "Técnico-científica"
    ],
    "Com registro": [
        (df["Contribuições de experiência e opinião"] != "Não informado").sum(),
        (df["Contribuições técnico - científicas"] != "Não informado").sum()
    ],
    "Sem informação": [
        (df["Contribuições de experiência e opinião"] == "Não informado").sum(),
        (df["Contribuições técnico - científicas"] == "Não informado").sum()
    ]
})

participacao_longa = participacao.melt(
    id_vars="Tipo de contribuição",
    var_name="Situação",
    value_name="Quantidade"
)

fig5 = px.bar(
    participacao_longa,
    x="Tipo de contribuição",
    y="Quantidade",
    color="Situação",
    barmode="group",
    text="Quantidade",
    title="Contribuições registradas na pesquisa"
)
fig5.show()



##Participação e contribuições

In [ ]:
display(Markdown("##Disponibilidade de documentos"))



documentos = [
    "Relatórios para a Sociedade",
    "Notas Técnicas",
    "Despacho",
    "Decisão do Ministério da Saúde - Relatório",
    "Relatório de recomendação Final"
]


documentos_dados = pd.DataFrame({
    "Documento": documentos,
    "Com registro": [
        (df[c] != "Não informado").sum() for c in documentos
    ],
    "Sem informação": [
        (df[c] == "Não informado").sum() for c in documentos
    ]
})

documentos_longo = documentos_dados.melt(
    id_vars="Documento",
    var_name="Situação",
    value_name="Quantidade"
)

fig6 = px.bar(
    documentos_longo,
    x="Documento",
    y="Quantidade",
    color="Situação",
    barmode="group",
    text="Quantidade",
    title="Presença de documentos nos registros"
)
fig6.update_layout(xaxis_tickangle=-25)
fig6.show()

##Disponibilidade de documentos

In [ ]:
display(Markdown("## Principais demandantes"))

demandantes = df[col_demandante].str.replace(
    r"\s+em\s+\d{2}/\d{2}/\d{4}.*$",
    "",
    regex=True
)




top_demandantes = (
    demandantes.value_counts()
    .head(10)
    .reset_index()
)
top_demandantes.columns = ["Demandante", "Quantidade"]

fig7 = px.bar(
    top_demandantes,
    x="Quantidade",
    y="Demandante",
    orientation="h",
    text="Quantidade",
    title="Até 10 demandantes mais frequentes"
)
fig7.update_layout(yaxis={"categoryorder": "total ascending"})
fig7.show()


## Principais demandantes

In [ ]:
display(Markdown("## Tabela resumida dos dados"))

resumo = df[[
    "Tecnologias e Diretrizes para COVID - 19",
    col_demandante,
    col_inicial,
    col_final,
    col_decisao
]]

display(resumo)


## Tabela resumida dos dados

,Tecnologias e Diretrizes para COVID - 19,Demandante,Análise inicial da Conitec,Análise Final da Conitec,Decisão do Ministério da Saúde
0,Nirmatrelvir/Ritonavir para o tratamento da Co...,"Secretaria de Ciência, Tecnologia e Inovação e...",Favorável à manutenção da incorporação,Favorável à manutenção da incorporação,Manutenção da incorporação
1,Vacina Pfizer/BioNTech para a imunização ativa...,Secretaria de Vigilância em Saúde em 18/10/2022,Favorável à incorporação,Favorável à incorporação,Incorporação
2,Molnupiravir para pacientes infectados por SAR...,Secretaria Executiva em 16/05/2022,Desfavorável à incorporação,Desfavorável à incorporação,Não incorporação
3,Tocilizumabe para tratamento de pacientes adul...,Ministro da Saúde em 14/04/2022,Favorável à incorporação,Favorável à incorporação,Incorporação
4,Cilgavimabe + tixagevimabe para profilaxia pré...,Secretaria Executiva em 11/04/2022 e pela Astr...,Desfavorável à incorporação,Desfavorável à incorporação,Não incorporação
5,Nirmatrelvir + ritonavir para o tratamento da ...,Secretaria Executiva em 05/04/2022,Favorável à incorporação,Favorável à incorporação,Incorporação
6,Baricitinibe para tratamento da Covid-19 em pa...,Elli Lilly do Brasil Ltda. em 24/01/2022,Favorável à incorporação,Favorável à incorporação,Incorporação
7,Casirivimabe/imdevimabe para pacientes infecta...,"Secretaria de Ciência, Tecnologia, Inovação e ...",Desfavorável à incorporação,Desfavorável à incorporação,Não incorporação
8,Banlanivimabe/etesevimabe para pacientes adult...,Ministro da Saúde em 18/05/2021,Desfavorável à incorporação,Desfavorável à incorporação,Não incorporação
9,Rendesivir para tratamento de pacientes com Co...,Gilead Sciences em 22/04/2021,Desfavorável à incorporação,Desfavorável à incorporação,Não incorporação


In [ ]:
display(Markdown("""
## Análise concluída

Foram gerados gráficos para observar:

- A distribuição das decisões do Ministério da Saúde
- A proporção de cada tipo de decisão
- A comparação entre as análises inicial e final da Conitec
- A presença de consultas públicas
- As contribuições de experiência, opinião e técnico-científicas
- A disponibilidade de documentos
- Os principais demandantes

**Observação:** os gráficos representam os registros existentes na planilha.
"Não informado" indica que a célula estava vazia ou marcada com "-".
"""))




## Análise concluída

Foram gerados gráficos para observar:

- A distribuição das decisões do Ministério da Saúde
- A proporção de cada tipo de decisão
- A comparação entre as análises inicial e final da Conitec
- A presença de consultas públicas
- As contribuições de experiência, opinião e técnico-científicas
- A disponibilidade de documentos
- Os principais demandantes

**Observação:** os gráficos representam os registros existentes na planilha.
"Não informado" indica que a célula estava vazia ou marcada com "-".
